# MedVision-AI Kaggle GPU Training Notebook

This notebook orchestrates Stage 1 and Stage 2 training for MedVision-AI on Kaggle.

## Workflow:
1. **Env Setup**: Verify Python, TensorFlow/Keras, GPU, and canonical paths
2. **Repo Sync**: Clone or update from GitHub; preserve artifacts
3. **Validation**: Distinguish Stage 1 resume vs Stage 2 source checkpoint validation
4. **Stage 2 Forensic** (Optional): FP32 vs mixed_float16 numerical comparison (10 batches max)
5. **Stage 1 Launcher** (Optional): Only if no valid Stage 1 checkpoint exists
6. **Stage 2 Launcher** (Active): Production fine-tuning with auto-resume
7. **Stage 2 Checkpoint Verification**: Confirm persistence
8. **Artifact Inventory**: List all outputs
9. **Final ZIP**: Package artifacts
10. **Download Link**: Get artifacts


In [ ]:
import os
import platform
import sys
from pathlib import Path
import subprocess

# Set Keras backend BEFORE any TensorFlow/Keras imports
os.environ['KERAS_BACKEND'] = 'tensorflow'

import tensorflow as tf
import keras

# Print environment
print('=' * 80)
print('ENVIRONMENT')
print('=' * 80)
print(f'Python version: {platform.python_version()}')
print(f'TensorFlow version: {tf.__version__}')
print(f'Keras version: {keras.__version__}')
print(f'GPU devices: {len(tf.config.list_physical_devices("GPU"))}')
for gpu in tf.config.list_physical_devices('GPU'):
    print(f'  - {gpu}')
print(f'KERAS_BACKEND: {os.environ.get("KERAS_BACKEND")}')

# Canonical paths
REPO = Path('/kaggle/working') / 'MedVision-AI'
OUTPUTS = Path('/kaggle/working') / 'medvision_outputs'
CHECKPOINTS = OUTPUTS / 'checkpoints'

print(f'\nCanonical paths:')
print(f'  REPO: {REPO}')
print(f'  OUTPUTS: {OUTPUTS}')
print(f'  CHECKPOINTS: {CHECKPOINTS}')
print('=' * 80)


In [ ]:
# Clone or update repository (preserve artifacts)
if REPO.exists():
    print(f'Repository already cloned at {REPO}')
    os.chdir(str(REPO))
    result = subprocess.run(['git', 'fetch', 'origin'], capture_output=True, text=True)
    print('git fetch origin:', result.stdout.strip() if result.stdout else '(up to date)')
    result = subprocess.run(['git', 'reset', '--hard', 'origin/main'], capture_output=True, text=True)
    print('git reset --hard origin/main:', result.stdout.strip())
else:
    print(f'Cloning repository to {REPO}...')
    REPO.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ['git', 'clone', 'https://github.com/SwastikPandey1024/MedVision-AI.git', str(REPO)],
        check=True
    )
    os.chdir(str(REPO))

# Install package in development mode (no dependencies, we have TensorFlow from Kaggle)
print('\nInstalling package: pip install -e . --no-deps')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps', '-q'], check=True)

# Print exact commit
result = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, cwd=str(REPO))
commit_sha = result.stdout.strip()
print(f'\nRepository at commit: {commit_sha}')
print('=' * 80)


In [ ]:
from medvision.config.settings import get_output_dir
from medvision.data.dataset import find_dataset_root, parse_rsna_manifest, create_real_rsna_dataset
from medvision.data.splits import create_patient_aware_splits
from medvision.models.trainer import (
    find_valid_resume_checkpoint,
    validate_stage2_source_checkpoint,
    resolve_stage2_source_checkpoint,
)

print('=' * 80)
print('VALIDATION: DATASET & CHECKPOINT PROVENANCE')
print('=' * 80)

# Find dataset
dataset_root = find_dataset_root()
print(f'Dataset root: {dataset_root}')
if dataset_root:
    manifest_path = dataset_root / 'manifest.csv'
    if manifest_path.exists():
        manifest = parse_rsna_manifest(str(manifest_path))
        print(f'RSNA manifest: {len(manifest)} images')
    else:
        print('Warning: manifest.csv not found in dataset root')
else:
    print('Error: RSNA dataset not found. Ensure Kaggle RSNA attachment is available.')

# Stage 1 Resume Checkpoint Validation
# Requires: optimizer state, exact epoch recovery, weights match, architecture match
print('\n' + '=' * 80)
print('STAGE 1 RESUME VALIDATION (Optimizer State Required)')
print('=' * 80)
stage1_ckpt = get_output_dir('checkpoints') / 'densenet121_stage1_best.keras'
try:
    result = find_valid_resume_checkpoint(str(stage1_ckpt), 'densenet121')
    print(f'Stage 1 checkpoint: {stage1_ckpt.name}')
    print(f'  Status: {result.status}')
    print(f'  Reason: {result.reason}')
except Exception as e:
    print(f'Stage 1 checkpoint validation: {e}')

# Stage 2 Source Checkpoint Validation
# Requires: model weights & architecture only (no optimizer state needed)
print('\n' + '=' * 80)
print('STAGE 2 SOURCE VALIDATION (Model Weights Only, Optimizer NOT Required)')
print('=' * 80)
stage2_src_ckpt = get_output_dir('checkpoints') / 'densenet121_stage1_best.keras'
try:
    source_result = resolve_stage2_source_checkpoint(str(stage1_ckpt), None, 'densenet121')
    print(f'Stage 2 source: {source_result.checkpoint_path}')
    print(f'  Status: {source_result.status}')
    print(f'  Model loaded: {source_result.model is not None}')
except Exception as e:
    print(f'Stage 2 source validation: {e}')
print('=' * 80)


## Stage 2 Forensic Experiment (Optional)

### Purpose
Compare FP32 vs mixed_float16 precision behaviors under identical Stage 2 conditions:
- Same validated Stage 1 checkpoint source
- Same dataset (10 train + 3 val batches max)
- Same optimizer (Adam, lr=1e-5, clipnorm=1.0)
- Same unfreeze strategy (top 20 layers, BatchNorm frozen)
- **Only** the precision policy differs

### Output
- FP32 pass/fail status
- mixed_float16 pass/fail status
- First bad batch (if NaN detected)
- First bad tensor (loss, weight, gradient, metric)
- Trainable layer/BatchNorm counts

### Expected Duration
~5-10 minutes on GPU (small batch count, no full training)

### Run Command
```bash
python scripts/train.py \\
  --mode full \\
  --stage stage2_forensic \\
  --batch-size 32 \\
  --epochs 1 \\
  --mixed-precision
```


In [ ]:
# OPTIONAL: Stage 2 Forensic - FP32 vs mixed_float16 comparison
# Uncomment and run if you want to diagnose numerical precision issues.
# Max 10 train + 3 val batches, no full training.

import subprocess
import sys
from pathlib import Path

# Uncomment to enable:
# ENABLE_FORENSIC = True
ENABLE_FORENSIC = False

if ENABLE_FORENSIC:
    os.chdir(str(REPO))
    print('=' * 80)
    print('STARTING STAGE 2 FORENSIC: FP32 vs mixed_float16')
    print('=' * 80)
    cmd = [
        sys.executable,
        'scripts/train.py',
        '--mode', 'full',
        '--stage', 'stage2_forensic',
        '--batch-size', '32',
        '--epochs', '1',
        '--mixed-precision',
    ]
    result = subprocess.run(cmd)
    sys.exit(result.returncode)
else:
    print('Stage 2 Forensic is DISABLED.')
    print('Set ENABLE_FORENSIC = True above to run the forensic comparison.')


## Stage 1 Launcher (Optional)

**Only run if a valid Stage 1 checkpoint does NOT already exist.**

Stage 1:
- Trains only the DenseNet121 classification head
- Freezes backbone (ImageNet pretrained weights)
- Outputs: `densenet121_stage1_best.keras` with full optimizer state
- Duration: ~4-8 hours on GPU

If a valid checkpoint exists, Stage 2 will auto-resume from it via `--auto-resume`.


In [ ]:
# OPTIONAL: Stage 1 launcher
# Only run if no valid Stage 1 checkpoint exists.
# Duration: ~4-8 hours on GPU

import subprocess
import sys

ENABLE_STAGE1 = False  # Set to True only if needed

if ENABLE_STAGE1:
    os.chdir(str(REPO))
    print('=' * 80)
    print('STARTING STAGE 1: DenseNet121 Head Training')
    print('=' * 80)
    cmd = [
        sys.executable,
        'scripts/train.py',
        '--mode', 'full',
        '--stage', 'stage1',
        '--batch-size', '32',
        '--epochs', '3',
        '--mixed-precision',
    ]
    result = subprocess.run(cmd)
    sys.exit(result.returncode)
else:
    print('Stage 1 is DISABLED (optional).')
    print('Set ENABLE_STAGE1 = True above only if no valid Stage 1 checkpoint exists.')


## Stage 2 Production Launcher (ACTIVE)

### Workflow
1. Auto-resumes from valid Stage 1 checkpoint (via `--auto-resume`)
2. Unfreezes top 20 DenseNet121 layers
3. Keeps BatchNorm frozen
4. Fine-tunes with optimizer: Adam (lr=1e-5, clipnorm=1.0)
5. Outputs: `densenet121_stage2_best.keras` (model weights only, no optimizer state)
6. Duration: ~2-4 hours on GPU for 3 epochs

### Auto-Resume Behavior
- Searches for `densenet121_stage1_best.keras`
- Validates Stage 1 checkpoint (optimizer state required)
- If invalid, Stage 1 must be rerun
- If valid, loads weights and proceeds immediately to Stage 2


In [ ]:
# PRODUCTION: Stage 2 Fine-Tuning Launcher (ACTIVE)
# This is the main training path.
# It auto-resumes from valid Stage 1 checkpoint.

import subprocess
import sys
from pathlib import Path

os.chdir(str(REPO))
print('=' * 80)
print('STARTING STAGE 2: DenseNet121 Fine-Tuning (Production)')
print('  Mode: full (Kaggle RSNA dataset)')
print('  Auto-resume: enabled')
print('  Mixed-precision: enabled')
print('  Epochs: 3')
print('  Batch-size: 32')
print('  Unfroze: top 20 layers, BatchNorm frozen')
print('=' * 80)

cmd = [
    sys.executable,
    'scripts/train.py',
    '--mode', 'full',
    '--stage', 'stage2',
    '--epochs', '3',
    '--batch-size', '32',
    '--mixed-precision',
    '--auto-resume',
]

result = subprocess.run(cmd)
sys.exit(result.returncode)


In [ ]:
from pathlib import Path
from medvision.models.trainer import verify_checkpoint_persistence

stage2_ckpt = CHECKPOINTS / 'densenet121_stage2_best.keras'

print('=' * 80)
print('STAGE 2 CHECKPOINT VERIFICATION')
print('=' * 80)

if stage2_ckpt.exists():
    print(f'Stage 2 checkpoint found: {stage2_ckpt}')
    print(f'  Size: {stage2_ckpt.stat().st_size / 1e6:.2f} MB')
    
    # Verify persistence
    try:
        verification_result = verify_checkpoint_persistence(str(stage2_ckpt))
        print(f'  Persistence check: {verification_result.status}')
        print(f'  Reason: {verification_result.reason}')
    except Exception as e:
        print(f'  Verification error: {e}')
else:
    print('Stage 2 checkpoint NOT found.')
    print(f'Expected path: {stage2_ckpt}')
    print('Ensure Stage 2 training completed successfully.')
print('=' * 80)


In [ ]:
import os
from pathlib import Path

print('=' * 80)
print('ARTIFACT INVENTORY')
print('=' * 80)

runtime_dir = Path('/kaggle/working/medvision_outputs')
if runtime_dir.exists():
    print(f'Runtime artifacts directory: {runtime_dir}\n')
    
    for root, dirs, files in os.walk(runtime_dir):
        level = root.replace(str(runtime_dir), '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        sub_indent = ' ' * 2 * (level + 1)
        for file in sorted(files):
            file_path = Path(root) / file
            size_mb = file_path.stat().st_size / 1e6
            print(f'{sub_indent}{file} ({size_mb:.2f} MB)')
else:
    print(f'Runtime directory not found: {runtime_dir}')

print('\n' + '=' * 80)


In [ ]:
import shutil
from pathlib import Path

runtime_dir = Path('/kaggle/working/medvision_outputs')
zip_path = Path('/kaggle/working/medvision_stage1_stage2_artifacts.zip')

print('=' * 80)
print('CREATE FINAL ARTIFACT ZIP')
print('=' * 80)

if runtime_dir.exists():
    print(f'Zipping {runtime_dir} -> {zip_path}...')
    shutil.make_archive(str(zip_path.with_suffix('')), 'zip', runtime_dir.parent, runtime_dir.name)
    zip_size_mb = zip_path.stat().st_size / 1e6
    print(f'Created: {zip_path} ({zip_size_mb:.2f} MB)')
else:
    print(f'No runtime directory found at {runtime_dir}')
print('=' * 80)


In [ ]:
from IPython.display import FileLink
from pathlib import Path

zip_path = Path('/kaggle/working/medvision_stage1_stage2_artifacts.zip')

if zip_path.exists():
    print('Download your artifacts:')
    display(FileLink(str(zip_path)))
else:
    print('No ZIP file found. Run the artifact packaging cell above first.')
